# Codon-conditioned backbone signal: robustness analysis

This notebook stress-tests the codon-pair rejections shown in Figure 1. For every pair
flagged significant by at least one two-sample statistic, it asks: how many of the most
influential observations would need to be removed before the rejection disappears
(adversarial breakdown-k), and does the same rejection survive under label-randomized
null controls that preserve every observation's (phi, psi)?

Everything below is derived live from the same `cc-pvals.csv` files Figure 1 loads --
re-running the upstream pipeline and then re-running this notebook regenerates every
number here with no manual step.

In [1]:
import sys
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
sys.path.insert(0, str(Path("../../scripts/pnas2026").resolve()))

from _pnas2026_common import LABELLED_TAGS, compute_bh_rejections, load_all_pvals
from _common import SEED

from pp5.dihedral import flat_torus_distance
from pp5.distributions.kde import gaussian_kernel
from pp5.stats.breakdown import breakdown_k_kde, breakdown_k_mmd
from pp5.stats.controls import (
    gen_aa_ss_control_replicates,
    gen_pooled_shuffle_replicates,
    null_control_summary,
)

FDR = 0.05
K_PERM = 5000  # permutations for real-pair baseline p-values and breakdown-k checkpoints
K_CTRL = 2000  # permutations for control-replicate baseline/breakdown checkpoints
N_CONTROL_REPLICATES = 30

# torus-W2 breakdown-k uses a separate, R-backed algorithm and is slow (the last full
# run over 6 pairs took ~50 minutes). Set True to recompute it live; when False, values
# are loaded from TORUS_CACHE_PATH, with graceful "not computed" handling for any
# candidate pair missing from that file.
RUN_TORUS_BREAKDOWN_LIVE = False
TORUS_CACHE_PATH = Path("../../out/pnas-2026-phase1-golden/robustness_torus_summary.csv").resolve()

# The 5 statistics with an adversarial breakdown-k procedure.
BREAKDOWN_ARMS = [
    "kde-l1(bw=10.0)",
    "kde-l1(bw=CV)",
    "mmd(unbiased)",
    "mmd(biased)",
    "w2torus(4-fixed)",
]
FAST_ARMS = [arm for arm in BREAKDOWN_ARMS if arm != "w2torus(4-fixed)"]

OUT_DIR = Path("../../out/pnas-2026/figs").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load p-values

Same registry and loader Figure 1 uses (`_pnas2026_common.py`), so this notebook can never
diverge on which runs or thresholds are in play.

In [2]:
df_all_pvals = load_all_pvals(LABELLED_TAGS)
df_rejections = compute_bh_rejections(df_all_pvals, fdr=FDR)
real_rejections = df_rejections[
    (df_rejections["codon_randomization"] == "none")
    & (df_rejections["stat_test"].isin(BREAKDOWN_ARMS))
].reset_index(drop=True)
real_rejections

,stat_test,codon_randomization,SS,n_hypotheses,fdr,bh_pvalue_threshold,n_rejected,rejected_pairs
0,w2torus(4-fixed),none,HELIX,87,0.05,0.002299,4,"[L-CTC:L-TTG, L-CTC:L-CTG, L-CTC:L-CTT, R-AGG:..."
1,w2torus(4-fixed),none,OTHER,87,0.05,0.000000,0,[]
2,w2torus(4-fixed),none,SHEET,87,0.05,0.000000,0,[]
3,w2torus(4-fixed),none,TURN,87,0.05,0.000575,1,[A-GCG:A-GCT]
4,kde-l1(bw=10.0),none,HELIX,87,0.05,0.001149,2,"[L-CTC:L-TTG, L-CTC:L-CTG]"
5,kde-l1(bw=10.0),none,OTHER,87,0.05,0.000000,0,[]
6,kde-l1(bw=10.0),none,SHEET,87,0.05,0.000000,0,[]
7,kde-l1(bw=10.0),none,TURN,87,0.05,0.001149,2,"[P-CCC:P-CCG, A-GCG:A-GCT]"
8,kde-l1(bw=CV),none,HELIX,87,0.05,0.000575,1,[L-CTC:L-TTG]
9,kde-l1(bw=CV),none,OTHER,87,0.05,0.000000,0,[]


## Load raw inputs

The pvals CSVs only carry aggregated statistics per codon pair; the breakdown-k and
provenance analysis below needs the underlying per-position data: the aggregated
dataset, the cross-validated KDE bandwidths (for the KDE-CV arm), and the raw
per-structure records (for PDB provenance).

In [3]:
DATASET_PATH = Path(
    "../../out/prec-collected/20211001_124553-aida-ex_EC-src_EC/results/"
    "pointwise_cdist-natcom/_intermediate_/dataset.csv"
).resolve()
CV_BANDWIDTH_PATH = Path("../../out/pnas-2026/bandwidth_cv/kernel_bandwidths.csv").resolve()
PROVENANCE_PATH = Path(
    "../../out/prec-collected/20211001_124553-aida-ex_EC-src_EC/data-precs.csv"
).resolve()

assert DATASET_PATH.is_file(), DATASET_PATH
assert CV_BANDWIDTH_PATH.is_file(), CV_BANDWIDTH_PATH
assert PROVENANCE_PATH.is_file(), PROVENANCE_PATH

df_dataset = pd.read_csv(DATASET_PATH)
if "AA" not in df_dataset.columns:
    df_dataset["AA"] = df_dataset["codon"].str.split("-").str[0]

df_cv_bandwidth = pd.read_csv(CV_BANDWIDTH_PATH).set_index(["codon", "ss"])["sigma_cv_deg"]

## Determine candidate pairs

A pair is a candidate for breakdown-k if at least one of the 5 statistics above rejects
it in the real data, at that statistic's own BH threshold. This replaces any fixed pair
list: a future rerun with different p-values changes the candidate set automatically.

In [4]:
candidate_pairs = sorted(
    {
        (row.SS, pair)
        for row in real_rejections.itertuples()
        for pair in row.rejected_pairs
    }
)
print(f"{len(candidate_pairs)} candidate pairs:")
for ss, pair in candidate_pairs:
    print(f"  {ss}  {pair}")

# Sanity check against the historical stress-test set documented in the revision plan
# (Section 4): the same union should still appear, since these pvals are the ones that
# set produced.
HISTORICAL_UNION = {
    ("HELIX", "L-CTC:L-TTG"),
    ("HELIX", "L-CTC:L-CTG"),
    ("HELIX", "L-CTC:L-CTT"),
    ("HELIX", "R-AGG:R-CGA"),
    ("TURN", "A-GCG:A-GCT"),
    ("TURN", "P-CCC:P-CCG"),
}
missing = HISTORICAL_UNION - set(candidate_pairs)
assert not missing, f"Historical stress-test pairs missing from the candidate set: {missing}"

6 candidate pairs:
  HELIX  L-CTC:L-CTG
  HELIX  L-CTC:L-CTT
  HELIX  L-CTC:L-TTG
  HELIX  R-AGG:R-CGA
  TURN  A-GCG:A-GCT
  TURN  P-CCC:P-CCG


## Shared helpers

`codon_angles` extracts one codon's (phi, psi) points (in radians) for a secondary-structure
class, plus the (unp_id, unp_idx) key of each point (needed later to trace adversarially-removed
points back to PDB structures). `k_grid_for` is the adversarial-removal checkpoint schedule,
used identically by every arm below.

In [5]:
def codon_angles(df: pd.DataFrame, ss: str, codon: str) -> tuple[np.ndarray, np.ndarray]:
    """Radian (phi, psi) points and (unp_id, unp_idx) keys for one (SS, codon) group.

    :param df: Dataset with columns `condition_group`, `codon`, `phi`, `psi`, `unp_id`,
        `unp_idx`.
    :param ss: Secondary-structure class, e.g. `"HELIX"`.
    :param codon: Codon label, e.g. `"L-CTC"`.
    :return: `(n, 2)` radian phi/psi array, `(n, 2)` object array of (unp_id, unp_idx).
    """
    sub = df[(df["condition_group"] == ss) & (df["codon"] == codon)]
    points = np.deg2rad(sub[["phi", "psi"]].to_numpy())
    keys = sub[["unp_id", "unp_idx"]].to_numpy()
    return points, keys


def k_grid_for(n_min: int) -> list[int]:
    """Adaptive breakdown-k checkpoint grid, capped so the smaller group keeps >= 2 points.

    :param n_min: Size of the smaller of the two groups being compared.
    :return: Sorted list of checkpoint values, each `<= n_min - 2`.
    """
    k_cap = min(40, max(5, int(np.ceil(0.05 * n_min))))
    grid = sorted({0, 1, 2, 3, 5, 8, 12, 20, k_cap})
    return [k for k in grid if k <= n_min - 2]

## Breakdown-k per statistic

For each candidate pair, under each of the 4 fast statistics (KDE-fix, KDE-CV, MMD-unbiased,
MMD-biased), compute the baseline p-value; if it clears that statistic's own BH threshold,
run the adversarial breakdown-k procedure (greedily remove the most influential remaining
observation, re-ranked each time, until the pair is no longer significant) and the same
statistic's AA+SS and pooled-shuffle null controls. Non-significant (pair, statistic)
combinations are recorded with `breakdown_k=None` and no controls, matching how they'd be
reported as "n.s." in the summary table later.

In [6]:
def baseline_pval(breakdown_fn, X: np.ndarray, Y: np.ndarray) -> tuple[float, float]:
    """Baseline (ddist, p) at k=0, via the same breakdown_fn used for the real computation.

    :param breakdown_fn: A `(X, Y, thresh, k_grid) -> (rows, breakdown_k, removed_log)`
        callable, e.g. as built by `build_kde_fix_breakdown_fn`.
    :param X: `(n1, 2)` radian phi/psi observations for group 1.
    :param Y: `(n2, 2)` radian phi/psi observations for group 2.
    :return: (ddist, p) at the full, unmodified sample.
    """
    # thresh=1.0 + k_grid=[0] guarantees the loop stops after computing exactly the
    # k=0 checkpoint, without ever attempting a removal.
    rows, _, _ = breakdown_fn(X, Y, thresh=1.0, k_grid=[0])
    return rows[0]["ddist"], rows[0]["p"]


# Shared accumulators, appended to by this task and Task 7 (torus arm).
breakdown_rows: list[dict] = []
p_trajectories: dict[tuple[str, str, str], list[dict]] = {}
removed_logs: dict[tuple[str, str, str], dict] = {}

In [7]:
NBINS, GRID_LOW, GRID_HIGH, DTYPE = 128, -np.pi, np.pi, np.float64
SIGMA_FIXED_RAD = np.deg2rad(10.0)


def build_kde_fix_breakdown_fn(sigma_rad: float, k_perm: int):
    """Build a `(X, Y, thresh, k_grid) -> ...` breakdown_fn for the fixed-bandwidth KDE-L1 arm.

    :param sigma_rad: Shared bandwidth (radians) for both groups.
    :param k_perm: Number of permutations for the p-value at each checkpoint --
        `K_PERM` for the real pair, `K_CTRL` for control replicates (control
        replicates don't need the same permutation count as the real pair, and
        using the smaller `K_CTRL` keeps the 60-replicates-per-arm control cost
        tractable).
    """

    def _fn(X, Y, thresh, k_grid):
        return breakdown_k_kde(
            X,
            Y,
            n_bins=NBINS,
            grid_low=GRID_LOW,
            grid_high=GRID_HIGH,
            dtype=DTYPE,
            sigma_x_rad=sigma_rad,
            sigma_y_rad=sigma_rad,
            thresh=thresh,
            k_grid=k_grid,
            k_perm=k_perm,
            k_min=k_perm,
            k_th=float("inf"),
            seed=SEED,
        )

    return _fn


def build_kde_cv_breakdown_fn(sigma_x_rad: float, sigma_y_rad: float, k_perm: int):
    """Build a `(X, Y, thresh, k_grid) -> ...` breakdown_fn for the per-codon CV-bandwidth KDE-L1 arm.

    :param sigma_x_rad: Group-1 (codon 1) bandwidth, radians.
    :param sigma_y_rad: Group-2 (codon 2) bandwidth, radians.
    :param k_perm: Number of permutations for the p-value at each checkpoint --
        `K_PERM` for the real pair, `K_CTRL` for control replicates.
    """

    def _fn(X, Y, thresh, k_grid):
        return breakdown_k_kde(
            X,
            Y,
            n_bins=NBINS,
            grid_low=GRID_LOW,
            grid_high=GRID_HIGH,
            dtype=DTYPE,
            sigma_x_rad=sigma_x_rad,
            sigma_y_rad=sigma_y_rad,
            thresh=thresh,
            k_grid=k_grid,
            k_perm=k_perm,
            k_min=k_perm,
            k_th=float("inf"),
            seed=SEED,
        )

    return _fn


def build_mmd_breakdown_fn(unbiased: bool, k_perm: int):
    """Build a `(X, Y, thresh, k_grid) -> ...` breakdown_fn for the MMD^2 arm.

    :param unbiased: If True, use the unbiased U-statistic; if False, the biased
        V-statistic.
    :param k_perm: Number of permutations for the p-value at each checkpoint --
        `K_PERM` for the real pair, `K_CTRL` for control replicates. MMD's
        breakdown-k rebuilds the full Gram matrix at every removal step (no
        shared precomputed representation, unlike the KDE arms), so this
        matters most here: at `K_PERM` every one of the 60 control replicates
        per (pair, arm) would re-run a full-cost permutation test, which is
        what makes `K_CTRL` (2000, vs. 5000) worth having as a distinct value.
    """

    def _fn(X, Y, thresh, k_grid):
        return breakdown_k_mmd(
            X,
            Y,
            similarity_fn=flat_torus_distance,
            kernel_fn=partial(gaussian_kernel, sigma=SIGMA_FIXED_RAD),
            unbiased=unbiased,
            thresh=thresh,
            k_grid=k_grid,
            k_perm=k_perm,
            k_min=k_perm,
            k_th=float("inf"),
            seed=SEED,
        )

    return _fn


def breakdown_fn_for_arm(arm: str, ss: str, codon1: str, codon2: str, k_perm: int):
    """Look up the `(X, Y, thresh, k_grid) -> ...` breakdown_fn for one of the 4 fast arms.

    The single entry point every fast arm goes through, so the compute loop below
    doesn't need one branch per arm family.

    :param arm: One of `FAST_ARMS`.
    :param ss: Secondary-structure class (needed to look up the CV bandwidth).
    :param codon1: First codon label (needed to look up the CV bandwidth).
    :param codon2: Second codon label (needed to look up the CV bandwidth).
    :param k_perm: Number of permutations to use -- pass `K_PERM` for the real
        pair, `K_CTRL` when building the breakdown_fn used for control replicates.
    :return: The breakdown_fn for this arm, specialized to this pair where relevant.
    """
    if arm == "kde-l1(bw=10.0)":
        return build_kde_fix_breakdown_fn(SIGMA_FIXED_RAD, k_perm=k_perm)
    if arm == "kde-l1(bw=CV)":
        sigma_x = np.deg2rad(float(df_cv_bandwidth.loc[(codon1, ss)]))
        sigma_y = np.deg2rad(float(df_cv_bandwidth.loc[(codon2, ss)]))
        return build_kde_cv_breakdown_fn(sigma_x, sigma_y, k_perm=k_perm)
    if arm == "mmd(unbiased)":
        return build_mmd_breakdown_fn(unbiased=True, k_perm=k_perm)
    if arm == "mmd(biased)":
        return build_mmd_breakdown_fn(unbiased=False, k_perm=k_perm)
    raise ValueError(f"No fast breakdown_fn for arm {arm!r} (torus is handled separately)")

In [8]:
for ss, pair in candidate_pairs:
    codon1, codon2 = pair.split(":")
    X, keys1 = codon_angles(df_dataset, ss, codon1)
    Y, keys2 = codon_angles(df_dataset, ss, codon2)
    n1, n2 = len(X), len(Y)
    k_grid = k_grid_for(min(n1, n2))

    for arm in FAST_ARMS:
        breakdown_fn = breakdown_fn_for_arm(arm, ss, codon1, codon2, k_perm=K_PERM)

        arm_row = real_rejections[
            (real_rejections["stat_test"] == arm) & (real_rejections["SS"] == ss)
        ]
        assert len(arm_row) == 1, f"Expected exactly one threshold row for {(arm, ss)}"
        thresh = float(arm_row["bh_pvalue_threshold"].iloc[0])

        ddist0, p0 = baseline_pval(breakdown_fn, X, Y)
        significant = p0 <= thresh

        row = dict(
            SS=ss,
            pair=pair,
            statistic=arm,
            n1=n1,
            n2=n2,
            ddist=ddist0,
            p=p0,
            significant=significant,
            breakdown_k=None,
            breakdown_pct=None,
            aass_frac_sig=None,
            aass_bk_median=None,
            aass_bk_max=None,
            aass_min_p0=None,
            pooled_frac_sig=None,
            pooled_bk_median=None,
            pooled_bk_max=None,
            pooled_min_p0=None,
        )

        if significant:
            rows, breakdown_k, removed_log = breakdown_fn(X, Y, thresh, k_grid)
            p_trajectories[(ss, pair, arm)] = rows
            removed_logs[(ss, pair, arm)] = dict(
                removed_log=removed_log, keys1=keys1, keys2=keys2, codon1=codon1, codon2=codon2
            )

            # Controls get their own breakdown_fn built at K_CTRL (2000), not K_PERM
            # (5000): 60 replicates/arm at the real pair's permutation count would
            # be far more expensive than the real computation itself, especially
            # for MMD, which rebuilds its Gram matrix from scratch at every step.
            # Default-arg `fn=control_breakdown_fn` captures *this* iteration's
            # closure value; without it, every replicate call below would see the
            # loop's *final* breakdown_fn instead of the one built for this arm.
            control_breakdown_fn = breakdown_fn_for_arm(arm, ss, codon1, codon2, k_perm=K_CTRL)
            aa_ss_ctrl = null_control_summary(
                gen_aa_ss_control_replicates(df_dataset, ss, codon1, codon2, N_CONTROL_REPLICATES, SEED),
                pval_fn=lambda X_, Y_, fn=control_breakdown_fn: baseline_pval(fn, X_, Y_),
                breakdown_fn=control_breakdown_fn,
                thresh=thresh,
                k_grid_fn=k_grid_for,
            )
            pooled_ctrl = null_control_summary(
                gen_pooled_shuffle_replicates(X, Y, N_CONTROL_REPLICATES, SEED + 13),
                pval_fn=lambda X_, Y_, fn=control_breakdown_fn: baseline_pval(fn, X_, Y_),
                breakdown_fn=control_breakdown_fn,
                thresh=thresh,
                k_grid_fn=k_grid_for,
            )

            row.update(
                breakdown_k=breakdown_k,
                breakdown_pct=(breakdown_k / min(n1, n2) * 100) if breakdown_k else None,
                aass_frac_sig=aa_ss_ctrl["frac_sig"],
                aass_bk_median=aa_ss_ctrl["bk_median"],
                aass_bk_max=aa_ss_ctrl["bk_max"],
                aass_min_p0=aa_ss_ctrl["min_p0"],
                pooled_frac_sig=pooled_ctrl["frac_sig"],
                pooled_bk_median=pooled_ctrl["bk_median"],
                pooled_bk_max=pooled_ctrl["bk_max"],
                pooled_min_p0=pooled_ctrl["min_p0"],
            )

        breakdown_rows.append(row)

pd.DataFrame(breakdown_rows)

,SS,pair,statistic,n1,n2,ddist,p,significant,breakdown_k,breakdown_pct,aass_frac_sig,aass_bk_median,aass_bk_max,aass_min_p0,pooled_frac_sig,pooled_bk_median,pooled_bk_max,pooled_min_p0
0,HELIX,L-CTC:L-CTG,kde-l1(bw=10.0),528,2812,0.096589,0.000800,True,1.0,0.189394,0.0,0.0,0.0,0.026987,0.0,0.0,0.0,0.012994
1,HELIX,L-CTC:L-CTG,kde-l1(bw=CV),528,2812,0.378161,0.399520,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HELIX,L-CTC:L-CTG,mmd(unbiased),528,2812,0.003353,0.002000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,HELIX,L-CTC:L-CTG,mmd(biased),528,2812,0.004428,0.002000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HELIX,L-CTC:L-CTT,kde-l1(bw=10.0),528,530,0.085733,0.053989,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,HELIX,L-CTC:L-CTT,kde-l1(bw=CV),528,530,0.288744,0.039792,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,HELIX,L-CTC:L-CTT,mmd(unbiased),528,530,0.002146,0.042392,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,HELIX,L-CTC:L-CTT,mmd(biased),528,530,0.003926,0.042392,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,HELIX,L-CTC:L-TTG,kde-l1(bw=10.0),528,599,0.125916,0.000600,True,8.0,1.515152,0.0,0.0,0.0,0.017491,0.0,0.0,0.0,0.071464
9,HELIX,L-CTC:L-TTG,kde-l1(bw=CV),528,599,0.305241,0.001000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### torus-W2 arm (flagged: live or cached)

torus-W2's breakdown-k uses a different, R-backed algorithm (recomputing the analytic
projected-Wasserstein test at each removal checkpoint via the vendored `torustest`), and it
is slow: the last full run over 6 pairs took roughly 50 minutes. Set `RUN_TORUS_BREAKDOWN_LIVE
= True` above to recompute it here; otherwise this section loads breakdown-k from
`TORUS_CACHE_PATH`, and any candidate pair not present in that file is recorded as
"not computed" rather than causing an error.

In [9]:
if RUN_TORUS_BREAKDOWN_LIVE:
    sys.path.insert(0, str(Path("../../scripts/pnas2026").resolve()))
    from robustness_torus import breakdown_torus, torus_pval

    for ss, pair in candidate_pairs:
        codon1, codon2 = pair.split(":")
        X, keys1 = codon_angles(df_dataset, ss, codon1)
        Y, keys2 = codon_angles(df_dataset, ss, codon2)
        n1, n2 = len(X), len(Y)
        k_grid = k_grid_for(min(n1, n2))

        arm_row = real_rejections[
            (real_rejections["stat_test"] == "w2torus(4-fixed)") & (real_rejections["SS"] == ss)
        ]
        assert len(arm_row) == 1
        thresh = float(arm_row["bh_pvalue_threshold"].iloc[0])

        stat0, p0 = torus_pval(X, Y)
        significant = p0 <= thresh

        row = dict(
            SS=ss,
            pair=pair,
            statistic="w2torus(4-fixed)",
            n1=n1,
            n2=n2,
            ddist=stat0,
            p=p0,
            significant=significant,
            breakdown_k=None,
            breakdown_pct=None,
            aass_frac_sig=None,
            aass_bk_median=None,
            aass_bk_max=None,
            aass_min_p0=None,
            pooled_frac_sig=None,
            pooled_bk_median=None,
            pooled_bk_max=None,
            pooled_min_p0=None,
            source="computed live",
        )

        if significant:
            rows, breakdown_k, removed_log = breakdown_torus(X, Y, thresh, k_grid)
            p_trajectories[(ss, pair, "w2torus(4-fixed)")] = rows
            removed_logs[(ss, pair, "w2torus(4-fixed)")] = dict(
                removed_log=removed_log, keys1=keys1, keys2=keys2, codon1=codon1, codon2=codon2
            )

            aa_ss_ctrl = null_control_summary(
                gen_aa_ss_control_replicates(df_dataset, ss, codon1, codon2, N_CONTROL_REPLICATES, SEED),
                pval_fn=torus_pval,
                breakdown_fn=breakdown_torus,
                thresh=thresh,
                k_grid_fn=k_grid_for,
            )
            pooled_ctrl = null_control_summary(
                gen_pooled_shuffle_replicates(X, Y, N_CONTROL_REPLICATES, SEED + 13),
                pval_fn=torus_pval,
                breakdown_fn=breakdown_torus,
                thresh=thresh,
                k_grid_fn=k_grid_for,
            )

            row.update(
                breakdown_k=breakdown_k,
                breakdown_pct=(breakdown_k / min(n1, n2) * 100) if breakdown_k else None,
                aass_frac_sig=aa_ss_ctrl["frac_sig"],
                aass_bk_median=aa_ss_ctrl["bk_median"],
                aass_bk_max=aa_ss_ctrl["bk_max"],
                aass_min_p0=aa_ss_ctrl["min_p0"],
                pooled_frac_sig=pooled_ctrl["frac_sig"],
                pooled_bk_median=pooled_ctrl["bk_median"],
                pooled_bk_max=pooled_ctrl["bk_max"],
                pooled_min_p0=pooled_ctrl["min_p0"],
            )

        breakdown_rows.append(row)
else:
    cached = pd.read_csv(TORUS_CACHE_PATH) if TORUS_CACHE_PATH.is_file() else pd.DataFrame()
    for ss, pair in candidate_pairs:
        cached_row = (
            cached[(cached.get("SS") == ss) & (cached.get("pair") == pair)]
            if len(cached)
            else pd.DataFrame()
        )
        if len(cached_row):
            r = cached_row.iloc[0]
            breakdown_k = r["breakdown_k"] if pd.notna(r["breakdown_k"]) else None
            n1, n2 = int(r["n1"]), int(r["n2"])
            row = dict(
                SS=ss,
                pair=pair,
                statistic="w2torus(4-fixed)",
                n1=n1,
                n2=n2,
                ddist=None,
                p=float(r["torus_p"]),
                significant=bool(r["torus_sig"]),
                breakdown_k=breakdown_k,
                breakdown_pct=(breakdown_k / min(n1, n2) * 100) if breakdown_k else None,
                aass_frac_sig=r.get("aass_frac_sig"),
                aass_bk_median=None,
                aass_bk_max=None,
                aass_min_p0=r.get("aass_min_p0"),
                pooled_frac_sig=r.get("pooled_frac_sig"),
                pooled_bk_median=None,
                pooled_bk_max=None,
                pooled_min_p0=r.get("pooled_min_p0"),
                source=f"cached ({TORUS_CACHE_PATH.name})",
            )
        else:
            row = dict(
                SS=ss,
                pair=pair,
                statistic="w2torus(4-fixed)",
                n1=None,
                n2=None,
                ddist=None,
                p=None,
                significant=None,
                breakdown_k=None,
                breakdown_pct=None,
                aass_frac_sig=None,
                aass_bk_median=None,
                aass_bk_max=None,
                aass_min_p0=None,
                pooled_frac_sig=None,
                pooled_bk_median=None,
                pooled_bk_max=None,
                pooled_min_p0=None,
                source="not computed -- set RUN_TORUS_BREAKDOWN_LIVE=True or rerun robustness_torus.py",
            )
        breakdown_rows.append(row)

df_breakdown_all = pd.DataFrame(breakdown_rows)
df_breakdown_all

,SS,pair,statistic,n1,n2,ddist,p,significant,breakdown_k,breakdown_pct,aass_frac_sig,aass_bk_median,aass_bk_max,aass_min_p0,pooled_frac_sig,pooled_bk_median,pooled_bk_max,pooled_min_p0,source
0,HELIX,L-CTC:L-CTG,kde-l1(bw=10.0),528,2812,0.096589,0.000800,True,1.0,0.189394,0.0,0.0,0.0,0.026987,0.0,0.0,0.0,0.012994,NaN
1,HELIX,L-CTC:L-CTG,kde-l1(bw=CV),528,2812,0.378161,0.399520,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,HELIX,L-CTC:L-CTG,mmd(unbiased),528,2812,0.003353,0.002000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,HELIX,L-CTC:L-CTG,mmd(biased),528,2812,0.004428,0.002000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HELIX,L-CTC:L-CTT,kde-l1(bw=10.0),528,530,0.085733,0.053989,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,HELIX,L-CTC:L-CTT,kde-l1(bw=CV),528,530,0.288744,0.039792,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,HELIX,L-CTC:L-CTT,mmd(unbiased),528,530,0.002146,0.042392,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,HELIX,L-CTC:L-CTT,mmd(biased),528,530,0.003926,0.042392,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,HELIX,L-CTC:L-TTG,kde-l1(bw=10.0),528,599,0.125916,0.000600,True,8.0,1.515152,0.0,0.0,0.0,0.017491,0.0,0.0,0.0,0.071464,NaN
9,HELIX,L-CTC:L-TTG,kde-l1(bw=CV),528,599,0.305241,0.001000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
